# Pass Selection Model Smoke Test

This notebook runs a 1-epoch GPU smoke test on the current production model using the Leverkusen StatsBomb data. The goal is only to verify that the integrated 15-channel pipeline trains without shape or runtime issues.

In [ ]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import ConcatDataset, DataLoader, random_split

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "soccermap").exists():
            return candidate
    raise RuntimeError("Could not find project root containing code/soccermap.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CODE_DIR = PROJECT_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from soccermap.dataset import PassDataset
from soccermap.expand import build_expanded_dfs, build_player_id_mapping
from soccermap.model import SoccerMapConfig, SoccerMapWithPlayerEmbed, pass_selection_teammate_kl_loss
from soccermap.statsbomb_io import load_events, load_lineups, load_threesixty

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_ROOT = PROJECT_ROOT / "data" / "leverkusen_data"
TEAM_NAME = "Bayer Leverkusen"
BATCH_SIZE = 16
VAL_SPLIT = 0.15
LEARNING_RATE = 1e-3
EPOCHS = 1

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook.")

DEVICE = torch.device("cuda")
print("device:", DEVICE)
print("gpu:", torch.cuda.get_device_name(0))
print("data root:", DATA_ROOT)


In [ ]:
def list_team_match_ids(data_root: Path, team_name: str):
    event_dir = data_root / "events"
    ts_dir = data_root / "three-sixty"
    lineups_dir = data_root / "lineups"

    ids = []
    for event_file in sorted(event_dir.glob("*.json")):
        match_id = event_file.stem
        if not (ts_dir / f"{match_id}.json").exists():
            continue
        if not (lineups_dir / f"{match_id}.json").exists():
            continue
        lineups = load_lineups(data_root, match_id)
        team_names = [row.get("team_name") for row in lineups if isinstance(row, dict)]
        if team_name in team_names:
            ids.append(match_id)
    return ids


match_ids = list_team_match_ids(DATA_ROOT, TEAM_NAME)
print(f"matched {len(match_ids)} Leverkusen matches")
print("first few:", match_ids[:8])

datasets = []
lineups_list = []
total_samples = 0

for match_id in match_ids:
    events = load_events(DATA_ROOT, match_id)
    threesixty = load_threesixty(DATA_ROOT, match_id)
    lineups = load_lineups(DATA_ROOT, match_id)
    expanded = build_expanded_dfs(events, threesixty, lineups)
    ds_mid = PassDataset(
        expanded.expanded_df,
        only_passes=True,
        team_filter=TEAM_NAME,
        compute_velocities=False,
    )
    if len(ds_mid) == 0:
        continue
    datasets.append(ds_mid)
    lineups_list.append(lineups)
    total_samples += len(ds_mid)
    print(f"match {match_id}: {len(ds_mid)} samples")

if not datasets:
    raise RuntimeError("No pass samples were built from the Leverkusen data.")

full_ds = ConcatDataset(datasets)
player_id_mapping = build_player_id_mapping(lineups_list, team_name=TEAM_NAME)
num_players = len(player_id_mapping)

n_total = len(full_ds)
n_val = max(1, int(n_total * VAL_SPLIT))
n_train = n_total - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=lambda batch: batch)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=lambda batch: batch)

sample = full_ds[0]
print("total samples:", n_total)
print("train / val:", n_train, n_val)
print("channels shape:", tuple(sample.channels.shape))
print("context shape:", tuple(sample.context_features.shape))
print("players in embedding map:", num_players)


In [ ]:
def batch_to_tensors(batch):
    channels = torch.stack([b.channels for b in batch]).to(DEVICE)
    dest = torch.tensor([b.dest_index for b in batch], dtype=torch.long, device=DEVICE)
    actor_ids = torch.tensor(
        [player_id_mapping.get(b.actor_player_name, 0) for b in batch],
        dtype=torch.long,
        device=DEVICE,
    )
    context = torch.stack([b.context_features for b in batch]).to(DEVICE)
    return channels, dest, actor_ids, context


context_dim = int(sample.context_features.shape[0])
model = SoccerMapWithPlayerEmbed(
    num_players=num_players,
    embed_dim=8,
    context_dim=context_dim,
    context_hidden_dim=16,
    context_embed_dim=8,
    cfg=SoccerMapConfig(),
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

first_batch = next(iter(train_dl))
channels, dest, actor_ids, context = batch_to_tensors(first_batch)
with torch.no_grad():
    logits = model(channels, actor_ids, context)
print("smoke-test batch:", tuple(channels.shape), tuple(logits.shape))

history = []
for epoch in range(EPOCHS):
    model.train()
    train_total = 0.0
    train_n = 0
    for batch in train_dl:
        channels, dest, actor_ids, context = batch_to_tensors(batch)
        logits = model(channels, actor_ids, context)
        loss = pass_selection_teammate_kl_loss(logits, dest, channels[:, 0])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_total += float(loss.item()) * len(batch)
        train_n += len(batch)

    model.eval()
    val_total = 0.0
    val_n = 0
    with torch.no_grad():
        for batch in val_dl:
            channels, dest, actor_ids, context = batch_to_tensors(batch)
            logits = model(channels, actor_ids, context)
            loss = pass_selection_teammate_kl_loss(logits, dest, channels[:, 0])
            val_total += float(loss.item()) * len(batch)
            val_n += len(batch)

    train_loss = train_total / max(train_n, 1)
    val_loss = val_total / max(val_n, 1)
    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_samples": train_n,
        "val_samples": val_n,
    })
    print(f"epoch {epoch + 1}/{EPOCHS} train_loss={train_loss:.6f} val_loss={val_loss:.6f}")

history_df = pd.DataFrame(history)
history_df
